# DocuMate RAG spike

Phase 3a: minimal end-to-end notebook to iterate the prompt before lifting the API.

**Steps:** load Chroma → embed query (with BGE prefix) → top-5 → build XML prompt → Claude Sonnet → parse citations → display.

Run from `backend/`:
```bash
uv run jupyter notebook notebooks/01_rag_spike.ipynb
```

Prereqs:
- `make ingest` has been run (collection 'fairwork' in data/chroma/).
- `.env` has a valid `ANTHROPIC_API_KEY`.

In [ ]:
import sys
from pathlib import Path

# Make `app`, `ingest` importable when running from notebooks/
BACKEND = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

from app.config import get_settings
from app.services.chat_service import ChatService
from app.services.generator import Generator
from app.services.retriever import Retriever
from ingest.chroma_store import ChromaStore

s = get_settings()
store = ChromaStore(s.chroma_persist_dir, s.chroma_collection)
retriever = Retriever(store)
generator = Generator(api_key=s.anthropic_api_key, model=s.anthropic_generator_model)
chat = ChatService(retriever=retriever, generator=generator, top_k=s.retrieval_top_k)

print(f'Collection: {s.chroma_collection!r}, chunks={store.count()}')
print(f'Generator: {s.anthropic_generator_model}')
print(f'Embedder:  {s.embedding_model}')

In [ ]:
TEST_QUESTIONS = [
    ('factual_lookup', 'How much notice do I need to give when resigning after 2 years of service?'),
    ('factual_lookup', 'Do casual employees get paid annual leave?'),
    ('multi_hop',      'Can I take both parental leave and personal leave in the same year?'),
    ('aggregation',    'What types of leave am I entitled to under the National Employment Standards?'),
    ('out_of_scope',   "What's the GST rate in Australia?"),
]

def show(category, question):
    print('=' * 80)
    print(f'[{category}] {question}')
    print('-' * 80)
    resp = chat.chat(question)
    print(f'Latency: retrieve={resp.latency_ms.retrieve}ms generate={resp.latency_ms.generate}ms total={resp.latency_ms.total}ms')
    print()
    print('Retrieved chunk IDs:')
    for cid in resp.retrieved_chunk_ids:
        print(f'  - {cid}')
    print()
    print('Answer:')
    print(resp.answer)
    print()
    print('Validated citations:')
    for c in resp.citations:
        print(f'  - {c.chunk_id}  {c.title}')
    print()

for cat, q in TEST_QUESTIONS:
    show(cat, q)

## What to look for

- Factual answers contain the right numbers and at least one citation that matches a retrieved chunk.
- Multi-hop combines two chunks; cited IDs include both.
- Aggregation produces a list, not prose.
- Out-of-scope returns the exact refusal phrase from `app/prompts/answer.py`.